In [ ]:
"""
DyS‑CNN: Dynamic S‑box CNN Encryption
"""

import os
import sys
import numpy as np
import time
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import drive

# CONFIGURACIÓN
drive.mount('/content/drive')

RUTA_IMAGENES = "/content/drive/My Drive/img_2026"
RUTA_SALIDA = "/content/drive/My Drive/imagenes_resultados_DySCNN_2026"
os.makedirs(RUTA_SALIDA, exist_ok=True)
print(f"Images will be saved in: {RUTA_SALIDA}")

# IMPORTAR TENSORFLOW

try:
    import tensorflow as tf
    from tensorflow.keras import layers, models
    TF_AVAILABLE = True
    print(" TensorFlow detected.")
except ImportError:
    print(" TensorFlow not installed. Run: pip install tensorflow")
    sys.exit(1)

# CARGA DE IMÁGENES

def cargar_imagenes_reales(ruta_base):
    imagenes = {}
    print(f"\n Loading images from: {ruta_base}")
    if not os.path.exists(ruta_base):
        raise FileNotFoundError(f"Folder {ruta_base} does not exist.")
    archivos = [f for f in os.listdir(ruta_base) if f.lower().endswith('.png')]
    if len(archivos) == 0:
        raise FileNotFoundError(f"No .png files found in {ruta_base}")
    for archivo in archivos:
        nombre = os.path.splitext(archivo)[0]
        ruta_completa = os.path.join(ruta_base, archivo)
        try:
            img = Image.open(ruta_completa).convert('RGB')
            imagenes[nombre] = np.array(img, dtype=np.uint8)
            print(f"Loaded: {nombre} ({archivo}) -> {imagenes[nombre].shape}")
        except Exception as e:
            print(f"Error loading {archivo}: {e}")
    print(f"Total images loaded: {len(imagenes)}")
    return imagenes


# CNN PARA GENERAR S‑BOX

def crear_modelo_cnn_sbox():
    model = models.Sequential([
        layers.Input(shape=(16, 16, 1)),
        layers.Conv2D(16, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dense(256, activation='linear')
    ])
    return model

def generar_sbox_cnn(seed, model=None):
    if model is None:
        model = crear_modelo_cnn_sbox()
    np.random.seed(seed)
    entrada = np.random.rand(1, 16, 16, 1).astype(np.float32)
    salida = model(entrada, training=False).numpy().flatten()
    sbox = np.argsort(salida).astype(np.uint8)
    return sbox

# DIFUSIÓN GLOBAL REVERSIBLE

def difusion_adelante(img):
    flat = img.flatten()
    acum = 0
    out = np.zeros_like(flat, dtype=np.uint8)
    for i in range(len(flat)):
        acum = (acum + int(flat[i])) % 256
        out[i] = acum
    return out.reshape(img.shape).astype(np.uint8)

def difusion_inversa(img):
    flat = img.flatten()
    acum_anterior = 0
    out = np.zeros_like(flat, dtype=np.uint8)
    for i in range(len(flat)):
        original = (int(flat[i]) - acum_anterior) % 256
        acum_anterior = int(flat[i])
        out[i] = original
    return out.reshape(img.shape).astype(np.uint8)

# RONDAS DE CIFRADO/DESCIFRADO

def ronda_cifrado(img, sbox, seed):
    h, w, c = img.shape
    img_sust = sbox[img]
    np.random.seed(seed)
    indices = np.random.permutation(h * w * c)
    flat = img_sust.flatten()
    img_perm = flat[indices].reshape(h, w, c)
    np.random.seed(seed)
    clave = np.random.randint(0, 256, (h, w, c), dtype=np.uint8)
    img_xor = np.bitwise_xor(img_perm, clave)
    img_dif = difusion_adelante(img_xor)
    return img_dif

def ronda_descifrado(img, sbox_inv, seed):
    h, w, c = img.shape
    img_xor = difusion_inversa(img)
    np.random.seed(seed)
    clave = np.random.randint(0, 256, (h, w, c), dtype=np.uint8)
    img_perm = np.bitwise_xor(img_xor, clave)
    np.random.seed(seed)
    indices = np.random.permutation(h * w * c)
    inv_indices = np.argsort(indices)
    flat = img_perm.flatten()
    img_sust = flat[inv_indices].reshape(h, w, c)
    img_desc = sbox_inv[img_sust]
    return img_desc

def cifrar_imagen_hibrido(img, sbox, seed1=42, seed2=43):
    img = img.astype(np.uint8)
    img = ronda_cifrado(img, sbox, seed1)
    img = ronda_cifrado(img, sbox, seed2)
    return img

def descifrar_imagen_hibrido(img_cifrada, sbox_inv, seed1=42, seed2=43):
    img = img_cifrada.astype(np.uint8)
    img = ronda_descifrado(img, sbox_inv, seed2)
    img = ronda_descifrado(img, sbox_inv, seed1)
    return img

# MÉTRICAS

def calcular_entropia(img):
    hist = np.histogram(img, bins=256, range=[0,255])[0]
    hist = hist / hist.sum()
    hist = hist[hist>0]
    return -np.sum(hist * np.log2(hist))

def calcular_correlacion_adyacentes(img, n=1000):
    if len(img.shape) == 3:
        img_gray = np.mean(img, axis=2).astype(np.uint8)
    else:
        img_gray = img
    h, w = img_gray.shape
    np.random.seed(42)

    r1 = np.random.randint(0, h-1, n)
    c1 = np.random.randint(0, w-1, n)
    corr_h = np.corrcoef(img_gray[r1, c1], img_gray[r1, c1+1])[0, 1]

    r2 = np.random.randint(0, h-1, n)
    c2 = np.random.randint(0, w-1, n)
    corr_v = np.corrcoef(img_gray[r2, c2], img_gray[r2+1, c2])[0, 1]

    r3 = np.random.randint(0, h-1, n)
    c3 = np.random.randint(0, w-1, n)
    corr_d = np.corrcoef(img_gray[r3, c3], img_gray[r3+1, c3+1])[0, 1]

    corr_h = 0 if np.isnan(corr_h) else corr_h
    corr_v = 0 if np.isnan(corr_v) else corr_v
    corr_d = 0 if np.isnan(corr_d) else corr_d

    return {'horizontal': corr_h, 'vertical': corr_v, 'diagonal': corr_d}

def calcular_npcr_uaci(img1, img2):
    diff = np.abs(img1.astype(np.int32) - img2.astype(np.int32))
    npcr = np.mean(diff > 0) * 100
    uaci = np.mean(diff / 255.0) * 100
    return npcr, uaci

def modificar_pixel_robusto(img):
    img_mod = img.copy()
    val_orig = int(img_mod[0,0,0])
    img_mod[0,0,0] = (val_orig + 128) % 256
    return img_mod

def aplicar_ruido_sal_pimienta(img, prob=0.01):
    img_noisy = img.copy()
    mask = np.random.random(img.shape[:2]) < prob
    sal = np.random.choice([0,255], size=img.shape[:2])
    img_noisy[mask] = np.stack([sal[mask]]*3, axis=-1)
    return img_noisy

def aplicar_ruido_gaussiano(img, sigma=10):
    ruido = np.random.normal(0, sigma, img.shape).astype(np.int32)
    img_noisy = np.clip(img.astype(np.int32) + ruido, 0, 255).astype(np.uint8)
    return img_noisy

def aplicar_recorte(img, porcentaje=0.125):
    h, w, c = img.shape
    crop_size = int(np.sqrt(porcentaje * h * w))
    img_crop = img.copy()
    img_crop[:crop_size, :crop_size, :] = 0
    return img_crop

# VISUALIZACIÓN MEJORADA (TEXTOS EN INGLÉS)

def visualizacion_completa(original, cifrada, descifrada, nombre, sbox_inv):
    """
    Generates a high-quality figure with all texts in English.
    """
    # Apply attacks
    cif_sp = aplicar_ruido_sal_pimienta(cifrada, prob=0.01)
    desc_sp = descifrar_imagen_hibrido(cif_sp, sbox_inv)
    cif_gauss = aplicar_ruido_gaussiano(cifrada, sigma=10)
    desc_gauss = descifrar_imagen_hibrido(cif_gauss, sbox_inv)
    cif_crop = aplicar_recorte(cifrada, porcentaje=0.125)
    desc_crop = descifrar_imagen_hibrido(cif_crop, sbox_inv)

    # Figure with better spacing
    fig = plt.figure(figsize=(20, 24))
    gs = fig.add_gridspec(4, 3, hspace=0.4, wspace=0.35)

    # Main title
    fig.suptitle(f"DyS‑CNN - {nombre} (512×512)", fontsize=18, y=0.98, fontweight='bold')

    # Professional color palette
    hist_colors = ('#D62828', '#2E86AB', '#F77F00')  # Red, Blue, Orange

    # === Row 1: Images ===
    for i, (img, title) in enumerate(zip([original, cifrada, descifrada],
                                         ['Original', 'Encrypted', 'Decrypted'])):
        ax = fig.add_subplot(gs[0, i])
        ax.imshow(img)
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.axis('off')

    # === Row 2: Histograms ===
    for i, (img, title) in enumerate(zip([original, cifrada, descifrada],
                                         ['Hist. Original', 'Hist. Encrypted', 'Hist. Decrypted'])):
        ax = fig.add_subplot(gs[1, i])
        for j, color in enumerate(hist_colors):
            hist = np.histogram(img[:,:,j], bins=256, range=[0,255])[0]
            ax.plot(hist, color=color, alpha=0.8, linewidth=1.5,
                   label=['Red', 'Green', 'Blue'][j])
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.set_xlabel('Intensity', fontsize=11)
        ax.set_ylabel('Frequency', fontsize=11)
        ax.tick_params(labelsize=10)
        if i == 0:
            ax.legend(fontsize=10)

    # === Row 3: Correlations (H, V, D) of the encrypted image ===
    directions = ['horizontal', 'vertical', 'diagonal']
    corr_titles = ['Horizontal', 'Vertical', 'Diagonal']
    for i, (direc, tit) in enumerate(zip(directions, corr_titles)):
        ax = fig.add_subplot(gs[2, i])
        if len(cifrada.shape) == 3:
            img_gray = np.mean(cifrada, axis=2).astype(np.uint8)
        else:
            img_gray = cifrada
        h, w = img_gray.shape
        np.random.seed(42)
        n = min(1000, h*w//4)

        if direc == 'horizontal':
            r = np.random.randint(0, h-1, n)
            c = np.random.randint(0, w-1, n)
            x = img_gray[r, c]
            y = img_gray[r, c+1]
            ylabel = 'Pixel at (i, j+1)'
        elif direc == 'vertical':
            r = np.random.randint(0, h-1, n)
            c = np.random.randint(0, w-1, n)
            x = img_gray[r, c]
            y = img_gray[r+1, c]
            ylabel = 'Pixel at (i+1, j)'
        else:
            r = np.random.randint(0, h-1, n)
            c = np.random.randint(0, w-1, n)
            x = img_gray[r, c]
            y = img_gray[r+1, c+1]
            ylabel = 'Pixel at (i+1, j+1)'

        corr = np.corrcoef(x, y)[0, 1] if len(x) > 1 else 0
        ax.scatter(x, y, s=0.8, alpha=0.6, color='#2E86AB')
        ax.set_title(f'Encrypted - Corr. {tit} (Coef: {corr:.4f})', fontsize=12, fontweight='bold')
        ax.set_xlabel('Pixel at (i, j)', fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.tick_params(labelsize=10)

    # === Row 4: Attacks ===
    attacks = [
        (desc_sp, f'Salt & Pepper\nerror: {np.mean(np.abs(original.astype(np.float64)-desc_sp.astype(np.float64))):.3f}'),
        (desc_gauss, f'Gaussian noise (σ=10)\nerror: {np.mean(np.abs(original.astype(np.float64)-desc_gauss.astype(np.float64))):.3f}'),
        (desc_crop, f'Cropping 12.5%\nerror: {np.mean(np.abs(original.astype(np.float64)-desc_crop.astype(np.float64))):.3f}')
    ]
    for i, (img, title) in enumerate(attacks):
        ax = fig.add_subplot(gs[3, i])
        ax.imshow(img)
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.axis('off')

    plt.subplots_adjust(top=0.93, bottom=0.05, left=0.05, right=0.95)
    return fig

# ANÁLISIS COMPLETO

def analisis_completo(imagenes, sbox, sbox_inv):
    resultados = {}
    for nombre, img in imagenes.items():
        print(f"\n Analysing: {nombre} ({img.shape})...")
        t0 = time.time()
        img_cif = cifrar_imagen_hibrido(img, sbox)
        t_cif = time.time() - t0

        img_mod = modificar_pixel_robusto(img)
        img_cif_mod = cifrar_imagen_hibrido(img_mod, sbox)
        img_cif_clave2 = cifrar_imagen_hibrido(img, sbox, seed1=100, seed2=200)

        t0 = time.time()
        img_desc = descifrar_imagen_hibrido(img_cif, sbox_inv)
        t_desc = time.time() - t0
        error = np.mean(np.abs(img.astype(np.float64) - img_desc.astype(np.float64)))

        ent = calcular_entropia(img_cif)
        corr = calcular_correlacion_adyacentes(img_cif)
        npcr_px, uaci_px = calcular_npcr_uaci(img_cif, img_cif_mod)
        npcr_cl, uaci_cl = calcular_npcr_uaci(img_cif, img_cif_clave2)

        # Robustness
        cif_sp = aplicar_ruido_sal_pimienta(img_cif, prob=0.01)
        desc_sp = descifrar_imagen_hibrido(cif_sp, sbox_inv)
        error_sp = np.mean(np.abs(img.astype(np.float64) - desc_sp.astype(np.float64)))

        cif_gauss = aplicar_ruido_gaussiano(img_cif, sigma=10)
        desc_gauss = descifrar_imagen_hibrido(cif_gauss, sbox_inv)
        error_gauss = np.mean(np.abs(img.astype(np.float64) - desc_gauss.astype(np.float64)))

        cif_crop = aplicar_recorte(img_cif, porcentaje=0.125)
        desc_crop = descifrar_imagen_hibrido(cif_crop, sbox_inv)
        error_crop = np.mean(np.abs(img.astype(np.float64) - desc_crop.astype(np.float64)))

        resultados[nombre] = {
            'entropia': ent,
            'corr_h': corr['horizontal'],
            'corr_v': corr['vertical'],
            'corr_d': corr['diagonal'],
            'npcr_pixel': npcr_px,
            'uaci_pixel': uaci_px,
            'npcr_clave': npcr_cl,
            'uaci_clave': uaci_cl,
            'error': error,
            't_cif': t_cif,
            't_desc': t_desc,
            'img_cif': img_cif,
            'img_desc': img_desc,
            'robustez': {
                'sal_pimienta': error_sp,
                'gaussiano': error_gauss,
                'recorte': error_crop
            }
        }
        print(f"   Entropy: {ent:.4f}")
        print(f"   NPCR (pixel): {npcr_px:.2f}% | UACI (pixel): {uaci_px:.2f}%")
        print(f"   Correlations: H={corr['horizontal']:.4f}, V={corr['vertical']:.4f}, D={corr['diagonal']:.4f}")
        print(f"   Decryption error: {error:.6f}")
        print(f"   Robustness SP: {error_sp:.4f} | Gauss: {error_gauss:.4f} | Cropping: {error_crop:.4f}")

    return resultados

def generar_reporte_completo(resultados):
    print("\n" + "="*110)
    print(" FINAL REPORT OF CRYPTOGRAPHIC METRICS (JCR STANDARDS)")
    print("="*110)
    print(f"{'Image':<14} {'Entropy':<10} {'Corr.H':<10} {'Corr.V':<10} {'Corr.D':<10} {'NPCR%':<8} {'UACI%':<8} {'Error':<10} {'Time(s)':<9}")
    print("-"*110)
    for nombre, d in resultados.items():
        print(f"{nombre:<14} {d['entropia']:<10.4f} {d['corr_h']:<10.4f} {d['corr_v']:<10.4f} {d['corr_d']:<10.4f} "
              f"{d['npcr_pixel']:<8.2f} {d['uaci_pixel']:<8.2f} {d['error']:<10.6f} {d['t_cif']:<9.4f}")
    print("-"*110)

    avg_ent = np.mean([d['entropia'] for d in resultados.values()])
    avg_corr_h = np.mean([d['corr_h'] for d in resultados.values()])
    avg_corr_v = np.mean([d['corr_v'] for d in resultados.values()])
    avg_corr_d = np.mean([d['corr_d'] for d in resultados.values()])
    avg_npcr = np.mean([d['npcr_pixel'] for d in resultados.values()])
    avg_uaci = np.mean([d['uaci_pixel'] for d in resultados.values()])
    avg_err = np.mean([d['error'] for d in resultados.values()])
    avg_t = np.mean([d['t_cif'] for d in resultados.values()])

    print("\n **OVERALL AVERAGES:**")
    print(f"  - Average entropy     : {avg_ent:.4f}  (Ideal: 8.0000)")
    print(f"  - Average Corr. H     : {avg_corr_h:.4f}  (Ideal: 0.0000)")
    print(f"  - Average Corr. V     : {avg_corr_v:.4f}  (Ideal: 0.0000)")
    print(f"  - Average Corr. D     : {avg_corr_d:.4f}  (Ideal: 0.0000)")
    print(f"  - Average NPCR (pixel): {avg_npcr:.2f}%  (Ideal: 99.61%)")
    print(f"  - Average UACI (pixel): {avg_uaci:.2f}%  (Ideal: 33.46%)")
    print(f"  - Decryption error    : {avg_err:.6f}   (Ideal: 0.0000)")
    print(f"  - Avg. encryption time: {avg_t:.4f} seconds")

    cumple_npcr = avg_npcr >= 99.5
    cumple_uaci = 33.0 <= avg_uaci <= 34.0
    cumple_ent = avg_ent >= 7.99
    print("\n **QUALITY VERDICT (for JCR):**")
    print(f"  - NPCR  {' PASS' if cumple_npcr else ' IMPROVABLE'} (>=99.5% required)")
    print(f"  - UACI  {' PASS' if cumple_uaci else ' IMPROVABLE'} (33.0%-34.0% required)")
    print(f"  - Entropy {' PASS' if cumple_ent else ' IMPROVABLE'} (>=7.99 required)")
    if cumple_npcr and cumple_uaci and cumple_ent:
        print("   The system is CRYPTOGRAPHICALLY ROBUST for JCR standards!")
    else:
        print("   Adjust the S‑box or diffusion to improve these metrics.")

    print("\n **ROBUSTNESS AGAINST ATTACKS (average error):**")
    ataques = ['sal_pimienta', 'gaussiano', 'recorte']
    names = ['Salt & pepper', 'Gaussian', 'Cropping']
    for ataque, name in zip(ataques, names):
        errores = [d['robustez'][ataque] for d in resultados.values()]
        print(f"  - {name}: average error = {np.mean(errores):.4f}")

# EJECUCIÓN PRINCIPAL

if __name__ == "__main__":
    print(" STARTING DYS-CNN WITH IMPROVED IMAGE QUALITY (ENGLISH TEXTS)...")

    try:
        imagenes = cargar_imagenes_reales(RUTA_IMAGENES)
    except (FileNotFoundError, RuntimeError) as e:
        print(e)
        sys.exit(1)

    print("\n Creating CNN to generate dynamic S‑box...")
    model_cnn = crear_modelo_cnn_sbox()
    semilla_sbox = 42
    sbox = generar_sbox_cnn(semilla_sbox, model_cnn)
    print(f" S‑box generated by CNN (first 10): {sbox[:10]}")
    sbox_inv = np.argsort(sbox).astype(np.uint8)
    print(f" Inverse S‑box (first 10): {sbox_inv[:10]}")

    resultados = analisis_completo(imagenes, sbox, sbox_inv)
    generar_reporte_completo(resultados)

    print("\n  Generating visualizations with JCR quality (600 DPI)...")
    for nombre, img in imagenes.items():
        img_cif = resultados[nombre]['img_cif']
        img_desc = resultados[nombre]['img_desc']
        fig = visualizacion_completa(img, img_cif, img_desc, nombre, sbox_inv)

        # Save as TIFF (uncompressed, 600 DPI) for publication
        ruta_tiff = os.path.join(RUTA_SALIDA, f"{nombre}_visualization_complete.tiff")
        fig.savefig(ruta_tiff, dpi=600, bbox_inches='tight', facecolor='white',
                    format='tiff', pil_kwargs={"compression": None})
        print(f" TIFF (600 DPI): {ruta_tiff}")

        # Save as PNG (300 DPI) for quick preview
        ruta_png = os.path.join(RUTA_SALIDA, f"{nombre}_visualization_complete.png")
        fig.savefig(ruta_png, dpi=300, bbox_inches='tight', facecolor='white')
        print(f" PNG (300 DPI): {ruta_png}")

        plt.show()
        plt.close(fig)

    # Reversibility check
    img_test = list(imagenes.values())[0]
    cif = cifrar_imagen_hibrido(img_test, sbox)
    des = descifrar_imagen_hibrido(cif, sbox_inv)
    max_err = np.max(np.abs(img_test.astype(np.int16) - des.astype(np.int16)))
    print(f"\n Reversibility: Max error = {max_err} (ideal 0)")
    print(f" EXECUTION COMPLETE. All images saved in: {RUTA_SALIDA}")

Output hidden; open in https://colab.research.google.com to view.